## Transfer Learning with TensorFlow - Fine Tuning

In [15]:
import tensorflow as tf
print(tf.__version__)

2.18.0


In [16]:
!wget https://raw.githubusercontent.com/ROARMarketingConcepts/Tensorflow-Deep-Learning/refs/heads/main/extras/helper_functions.py

--2025-03-04 13:31:29--  https://raw.githubusercontent.com/ROARMarketingConcepts/Tensorflow-Deep-Learning/refs/heads/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10245 (10K) [text/plain]
Saving to: ‘helper_functions.py.2’

helper_functions.py 100%[===================>]  10.00K  --.-KB/s    in 0s      

2025-03-04 13:31:29 (22.6 MB/s) - ‘helper_functions.py.2’ saved [10245/10245]



In [17]:
from helper_functions import unzip_data,walk_through_dir

#### We will leverage pre-trained models in `tf.keras.applications` to apply to image classification use cases.

In [18]:
# Get 10% of the data of the 10 classes
!wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip

unzip_data("10_food_classes_10_percent.zip")

--2025-03-04 13:31:33--  https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.137.207, 142.250.101.207, 142.251.2.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.137.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 168546183 (161M) [application/zip]
Saving to: ‘10_food_classes_10_percent.zip.3’

10_food_classes_10_ 100%[===================>] 160.74M   113MB/s    in 1.4s    

2025-03-04 13:31:35 (113 MB/s) - ‘10_food_classes_10_percent.zip.3’ saved [168546183/168546183]



In [19]:
walk_through_dir("10_food_classes_10_percent")

There are 2 directories and 0 images in '10_food_classes_10_percent'.
There are 10 directories and 0 images in '10_food_classes_10_percent/test'.
There are 0 directories and 250 images in '10_food_classes_10_percent/test/ramen'.
There are 0 directories and 250 images in '10_food_classes_10_percent/test/steak'.
There are 0 directories and 250 images in '10_food_classes_10_percent/test/chicken_wings'.
There are 0 directories and 250 images in '10_food_classes_10_percent/test/ice_cream'.
There are 0 directories and 250 images in '10_food_classes_10_percent/test/grilled_salmon'.
There are 0 directories and 250 images in '10_food_classes_10_percent/test/sushi'.
There are 0 directories and 250 images in '10_food_classes_10_percent/test/pizza'.
There are 0 directories and 250 images in '10_food_classes_10_percent/test/chicken_curry'.
There are 0 directories and 250 images in '10_food_classes_10_percent/test/fried_rice'.
There are 0 directories and 250 images in '10_food_classes_10_percent/tes

Now we've got some image data, we need a way of loading it into a TensorFlow compatible format.

Previously, we've used the `ImageDataGenerator` class. However, as of August 2023, this class is deprecated and isn't recommended for future usage (it's too slow). Because of this, we'll move onto using `tf.keras.utils.image_dataset_from_directory()`.

This method expects image data in the following file format:

`main_directory/`

`...class_a/`

`......a_image_1.jpg`

`......a_image_2.jpg`

`...class_b/`

`......b_image_1.jpg`

`......b_image_2.jpg`



In [20]:
# Setup data inputs

train_dir = "10_food_classes_10_percent/train/"
test_dir = "10_food_classes_10_percent/test/"

IMG_SIZE = (224, 224) # define image size

train_data_10_percent = tf.keras.preprocessing.image_dataset_from_directory(directory=train_dir,
                                                                            image_size=IMG_SIZE,
                                                                            label_mode="categorical", # what type are the labels?
                                                                            batch_size=32) # batch_size is 32 by default, this is generally a good number

test_data_10_percent = tf.keras.preprocessing.image_dataset_from_directory(directory=test_dir,
                                                                           image_size=IMG_SIZE,
                                                                           label_mode="categorical")

Found 750 files belonging to 10 classes.
Found 2500 files belonging to 10 classes.


In [22]:
train_data_10_percent

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10), dtype=tf.float32, name=None))>

In [24]:
train_data_10_percent.class_names

['chicken_curry',
 'chicken_wings',
 'fried_rice',
 'grilled_salmon',
 'hamburger',
 'ice_cream',
 'pizza',
 'ramen',
 'steak',
 'sushi']

In [25]:
# See an example batch of data.  Notice that the labels are OneHotEncoded!

for images, labels in train_data_10_percent.take(1):
  print(images, labels)

tf.Tensor(
[[[[2.11891266e+02 1.94891266e+02 1.74891266e+02]
   [2.11830032e+02 1.94830032e+02 1.74830032e+02]
   [2.07847260e+02 1.91061539e+02 1.70418686e+02]
   ...
   [7.10375443e+01 5.64339676e+01 3.03116417e+01]
   [3.17361736e+01 2.58258266e+01 1.69959351e-01]
   [2.57120876e+01 2.57051525e+01 5.64173365e+00]]

  [[2.12647003e+02 1.95647003e+02 1.75647003e+02]
   [2.09463333e+02 1.92463333e+02 1.72463333e+02]
   [2.04590881e+02 1.87805176e+02 1.67162308e+02]
   ...
   [1.53454147e+02 1.38241211e+02 1.08501801e+02]
   [2.71924438e+01 1.87986774e+01 1.16403365e+00]
   [2.07422981e+01 1.84241009e+01 2.84307122e+00]]

  [[2.09611618e+02 1.92611618e+02 1.72611618e+02]
   [2.05397324e+02 1.88397324e+02 1.68397324e+02]
   [2.04897003e+02 1.88111282e+02 1.67468430e+02]
   ...
   [2.12050705e+02 1.94559372e+02 1.64001404e+02]
   [2.65821533e+01 1.51029434e+01 5.72808743e-01]
   [2.82387295e+01 2.24144402e+01 6.79186046e-01]]

  ...

  [[2.10540237e+02 1.89540237e+02 1.60600830e+02]
   [2

### Build the first model, `model_0`.

In [26]:
# 1. Create base model with tf.keras.applications
base_model = tf.keras.applications.efficientnet_v2.EfficientNetV2B0(include_top=False)

# OLD
# base_model = tf.keras.applications.EfficientNetB0(include_top=False)

# 2. Freeze the base model (so the pre-learned patterns remain)
base_model.trainable = False

# 3. Create inputs into the base model
inputs = tf.keras.layers.Input(shape=(224, 224, 3), name="input_layer")

# 4. If using ResNet50V2, add this to speed up convergence, remove for EfficientNetV2
# x = tf.keras.layers.experimental.preprocessing.Rescaling(1./255)(inputs)

# 5. Pass the inputs to the base_model (note: using tf.keras.applications, EfficientNetV2 inputs don't have to be normalized)
x = base_model(inputs)
# Check data shape after passing it to base_model
print(f"Shape after base_model: {x.shape}")

# 6. Average pool the outputs of the base model (aggregate all the most important information, reduce number of computations)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling_layer")(x)
print(f"After GlobalAveragePooling2D(): {x.shape}")

# 7. Create the output activation layer
outputs = tf.keras.layers.Dense(10, activation="softmax", name="output_layer")(x)

# 8. Combine the inputs with the outputs into a model
model_0 = tf.keras.Model(inputs, outputs)

# 9. Compile the model
model_0.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(),
              metrics=["accuracy"])

# 10. Fit the model (we use less steps for validation so it's faster)
history_10_percent = model_0.fit(train_data_10_percent,
                                 epochs=5,
                                 steps_per_epoch=len(train_data_10_percent),
                                 validation_data=test_data_10_percent,
                                 # Go through less of the validation data so epochs are faster (we want faster experiments!)
                                 validation_steps=int(0.25 * len(test_data_10_percent)),
                                 # Track our model's training logs for visualization later
                                 callbacks=[create_tensorboard_callback("transfer_learning", "10_percent_feature_extract")])


24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Shape after base_model: (None, 7, 7, 1280)
After GlobalAveragePooling2D(): (None, 1280)
Saving TensorBoard log files to: transfer_learning/10_percent_feature_extract/20250304-134837
Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 91s 3s/step - accuracy: 0.2207 - loss: 2.1872 - val_accuracy: 0.7122 - val_loss: 1.3841
Epoch 2/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 77s 3s/step - accuracy: 0.7370 - loss: 1.2552 - val_accuracy: 0.7911 - val_loss: 0.9317
Epoch 3/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 82s 3s/step - accuracy: 0.8061 - loss: 0.9135 - val_accuracy: 0.8438 - val_loss: 0.7238
Epoch 4/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 83s 4s/step - accuracy: 0.8277 - loss: 0.7506 - val_accuracy: 0.8618 - val_loss: 0.6184
Epoch 5/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 68s 3s/step - accuracy: 0.8624 - loss: 0.6180 - val_accuracy: 0.8701 - val_loss: 0.5662
